### Coleta de dados de Temperatura, Umidade e Precipitação

Será utilizado o dataset derived-era5-single-levels-daily-statistics do ERA5

Documentação em:
https://cds.climate.copernicus.eu/datasets/derived-era5-single-levels-daily-statistics?tab=documentation

Os dados extraídos:
<pre>
- Umidade   -> variável 2m_dewpoint_temperature (ponto de orvalho),
               O processo de conversão para percentual será detalhado no passo que executa a conversão
</pre>

Os dados serão coletados por Ano e Mês 

Os dados requisitados estão no retangulo geográfico geográfico [6, -74, -34, -35] -> [Norte, Oeste, Sul, Leste] em graus onde está o Brasil


In [ ]:
import cdsapi
import os
import xarray as xr

In [ ]:
import os, sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [ ]:
# Cria a conexão Spark

# 1. Remove qualquer barreira de proxy local que jogue o tráfego para a rede da empresa
os.environ.pop('HTTP_PROXY', None)
os.environ.pop('HTTPS_PROXY', None)
os.environ.pop('http_proxy', None)
os.environ.pop('https_proxy', None)

# 2. Garante que o Spark use o Python correto do venv
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# 3. Força o IP local estrito
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

# 4. Inicializa configurando a autenticação local do Worker
spark = SparkSession.builder \
    .appName("TesteLocal") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.network.auth.enabled", "false") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()


spark.sparkContext.setLogLevel("ERROR")


In [ ]:
# Os arquivos utilizados durante o processamento serão removidos no final do notebook
remover_arquivos = []

Requisição dos dados da variável derived-era5-single-levels-daily-statistics do ERA5 utilizando a biblioteca cdsapi

In [5]:
dataset = "derived-era5-single-levels-daily-statistics"
request = {
    "product_type": "reanalysis",
    "variable": [
        "2m_dewpoint_temperature"
    ],
    "year": "2025",
    "month": ["01"],
    "day": ["01", "02", "03",
            "04", "05", "06",
            "07", "08", "09",
            "10", "11", "12",
            "13", "14", "15"
    ],
    "daily_statistic": "daily_mean",
    "time_zone": "utc-03:00",
    "frequency": "1_hourly",
    "area": [6, -74, -34, -35] # Retangulo definido por [Norte, Oeste, Sul, Leste] em graus onde está o Brasil
}

# Informações de autenticação estão em:
# C:\Users\DRT90628\.ecmwfdatastoresrc
# *** Criar um novo contrato de autenticação deverá ser criado usando um usuário de serviços do Einstein

client = cdsapi.Client(
    url = os.getenv("ECMWF_DATASTORES_URL"),
    key = os.getenv("ECMWF_DATASTORES_KEY"),
)

ret_download = client.retrieve(dataset, request).download()
remover_arquivos.append(ret_download)

print(f"Download completed: {ret_download}")

2026-07-21 16:41:43,716 WARNING [2026-06-08T00:00:00] An issue with the following parameters has been identified:

- `maximum_2m_temperature_since_previous_post_processing`
- `minimum_2m_temperature_since_previous_post_processing`
- `10m_wind_gust_since_previous_post_processing`
- `maximum_total_precipitation_rate_since_previous_post_processing`
- `minimum_total_precipitation_rate_since_previous_post_processing`

Please refer to the documented [Known issue](https://confluence.ecmwf.int/display/CKB/ERA5+family+post-processed+daily+statistics+documentation#ERA5familypostprocesseddailystatisticsdocumentation-KnownIssues) for details. **Data downloaded for the parameters listed above should not be used.** A fix is to be released as soon as possible. Please [follow updates on this topic on our Forum announcement](https://forum.ecmwf.int/t/issue-affecting-some-parameters-from-the-era5-post-processed-daily-statistics-on-single-levels/15057).
2026-07-21 16:41:43,717 INFO Request ID is 31bc5a88

Download completed: 48595421830250f6217c0f1557b75ae6.nc


Esta função ira converter os dados dos arquivos .nc para o format Dask para então converter para Dataframe Spark <br>
Isso deixa o processamento em paralelo e será importante para processamento de grandes volumes (1 ano com todos os meses e dias)

In [6]:
def convert_netcdf4_Spark(file_name):
    with xr.open_dataset(f"C:\\Marco Conti\\Projetos\\MAIS-v2\\Ondas_Calor\\{file_name}"
                        ,engine="netcdf4"
                        ,chunks={"time": 365
                                ,"latitude": 100
                                ,"longitude": 100 }
                        ) as ds:
        
        # Transforma o Dataset em um Spark Dataframe
        df_dask   = ds.to_dask_dataframe()
        df_dask_c = df_dask.compute()
        df_spark  = spark.createDataFrame(df_dask_c)    
    return df_spark

In [ ]:
df_ponto_orvalho = convert_netcdf4_Spark(ret_download)

C:\Users\DRT90628\AppData\Local\Temp\ipykernel_8548\854711052.py:2: UserWarning: The specified chunks separate the stored chunks along dimension "latitude" starting at index 100. This could degrade performance. Instead, consider rechunking after loading.
  with xr.open_dataset(f"C:\\Marco Conti\\Projetos\\MAIS-v2\\Ondas_Calor\\{file_name}"
C:\Users\DRT90628\AppData\Local\Temp\ipykernel_8548\854711052.py:2: UserWarning: The specified chunks separate the stored chunks along dimension "longitude" starting at index 100. This could degrade performance. Instead, consider rechunking after loading.
  with xr.open_dataset(f"C:\\Marco Conti\\Projetos\\MAIS-v2\\Ondas_Calor\\{file_name}"
c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Marco Conti\Projetos\MAIS-v2\

root
 |-- valid_time: timestamp (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- number: long (nullable = true)
 |-- d2m: float (nullable = true)

+-------------------+--------+---------+------+---------+
|         valid_time|latitude|longitude|number|      d2m|
+-------------------+--------+---------+------+---------+
|2025-01-01 00:00:00|     6.0|    -74.0|     0|292.62863|
|2025-01-01 00:00:00|     6.0|   -73.75|     0|288.86423|
|2025-01-01 00:00:00|     6.0|    -73.5|     0| 287.5074|
|2025-01-01 00:00:00|     6.0|   -73.25|     0|285.55304|
|2025-01-01 00:00:00|     6.0|    -73.0|     0|282.10544|
|2025-01-01 00:00:00|     6.0|   -72.75|     0| 280.8627|
|2025-01-01 00:00:00|     6.0|    -72.5|     0|283.05646|
|2025-01-01 00:00:00|     6.0|   -72.25|     0|286.76422|
|2025-01-01 00:00:00|     6.0|    -72.0|     0|290.75284|
|2025-01-01 00:00:00|     6.0|   -71.75|     0|292.67944|
|2025-01-01 00:00:00|     6.0|    -71.5|    

In [8]:
drop_cols = ["valid_time", "number", "d2m"]
df_ponto_orvalho = \
    (df_ponto_orvalho
        .withColumns({"indicador": F.lit("ponto_orvalho")
                            ,"valor": (F.col("d2m") - F.lit(273.15)).cast('double')
                            ,"unidade_medida": F.lit("celsius")
                            ,"data_medicao": F.col("valid_time").cast("date")}
                            )
            .drop(*drop_cols)
    )

df_ponto_orvalho.printSchema()
df_ponto_orvalho.show()

root
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- indicador: string (nullable = false)
 |-- valor: double (nullable = true)
 |-- unidade_medida: string (nullable = false)
 |-- data_medicao: date (nullable = true)

+--------+---------+-------------+------------------+--------------+------------+
|latitude|longitude|    indicador|             valor|unidade_medida|data_medicao|
+--------+---------+-------------+------------------+--------------+------------+
|     6.0|    -74.0|ponto_orvalho|19.478631591796898|       celsius|  2025-01-01|
|     6.0|   -73.75|ponto_orvalho|15.714227294921898|       celsius|  2025-01-01|
|     6.0|    -73.5|ponto_orvalho|14.357385253906273|       celsius|  2025-01-01|
|     6.0|   -73.25|ponto_orvalho|12.403039550781273|       celsius|  2025-01-01|
|     6.0|    -73.0|ponto_orvalho| 8.955438232421898|       celsius|  2025-01-01|
|     6.0|   -72.75|ponto_orvalho| 7.712701416015648|       celsius|  2025-01-01|
|     6

In [ ]:
        # "source_file": "2m_dewpoint_temperature_stream-oper_daily-mean.nc",
        # "data_variable": "d2m",
        # "indicador": "ponto_orvalho",
        # "df_name": "df_ponto_orvalho",
        # "target_file": "ponto_orvalho.csv",
        # "unidade_original": "kelvin",
        # "unidade_destino": "celsius"

A seguir os 3 arquivos descompactados serão convertidos para um Dataframe Spark, cada dataframe receberá novas colunas (indicados e unidade de medida) <br>
e os valores serão convertidos de acordo com a necessidade de uso (ver dicionário acima)

In [ ]:

# rec = 0
# for file in zip_ref.filelist:


#      print("-"*100, "\n"
#          ,"File name: ", file.filename
#          ," Data variable:", get_file_attribute(file.filename, "data_variable")
#          ," - Target_file: ", get_file_attribute(file.filename, "target_file"))
    


#      # Convert os dados para um Dataframe Spark 
#      df_Spark = convert_netcdf4_Spark(file.filename)

#      data_variable   = get_file_attribute(file.filename, "data_variable")
#      unidade_medida  = get_file_attribute(file.filename, "unidade_destino")
#      nome_indicador  = get_file_attribute(file.filename, "indicador")
#      df_name         = get_file_attribute(file.filename, "df_name")

#      # Criação e remoção de colunas e conversão para unidades de medidas a serem usadas pelo MAIS
#      drop_cols = ["valid_time", "number"]
#      if data_variable in ('d2m', 't2m'):
#           drop_cols.append(data_variable)
#           globals()[df_name] = \
#                (df_Spark.withColumns({"indicador": F.lit(nome_indicador)
#                                      ,"valor": (F.col(data_variable) - F.lit(273.15)).cast('double')
#                                      ,"unidade_medida": F.lit(unidade_medida)
#                                      ,"data_medicao": F.col("valid_time").cast("date")}
#                                      )
#                         .drop(*drop_cols)
#                )
#      elif data_variable == "tp":
#           drop_cols.append(data_variable)
#           globals()[df_name] = \
#                (df_Spark.withColumns({"indicador": F.lit(nome_indicador)
#                                      ,"valor": (F.col(data_variable) * F.lit(1000)).cast('double')
#                                      ,"unidade_medida": F.lit(unidade_medida)
#                                      ,"data_medicao": F.col("valid_time").cast("date")}
#                                      )
#                         .drop(*drop_cols)
#                )

#      remover_arquivos.append(file.filename)

Faz a junção dos dados de temperatra e ponto de orvalho para calcular o percentual da umidade:

In [ ]:
df_temperatura = spark.read.parquet(r"C:\Marco Conti\Projetos\MAIS-v2\dados\ERA5-temperaturas\ERA5_temperatura.parquet")
print("Temperatura: ", df_temperatura.count())          # 783.587
print("Ponto de orvalho:", df_ponto_orvalho.count())    # 379.155

Temperatura:  783587
Ponto de orvalho: 379155


In [10]:
df_temp_ponto_orvalho = \
    (df_temperatura.alias('t')
        .join(df_ponto_orvalho.alias('p')
             ,((F.col('t.data_medicao') == F.col('p.data_medicao')) & 
               (F.col("t.latitude")     == F.col("p.latitude")) & 
               (F.col("t.longitude")    == F.col("p.longitude")))
             ,'inner')
        .select('t.data_medicao'
               ,'t.latitude'
               ,'t.longitude'
               ,F.col('t.valor').alias('temperatura_celsius')
               ,F.col('p.valor').alias('temp_ponto_orvalho_celsius'))
    )


print("Join:", df_temp_ponto_orvalho.count())

Join: 379155


In [11]:
df_temp_ponto_orvalho.limit(10).show(truncate=False)

+------------+--------+---------+-------------------+--------------------------+
|data_medicao|latitude|longitude|temperatura_celsius|temp_ponto_orvalho_celsius|
+------------+--------+---------+-------------------+--------------------------+
|2025-01-01  |6.0     |-74.0    |21.317346191406273 |19.478631591796898        |
|2025-01-01  |6.0     |-73.75   |17.715447998046898 |15.714227294921898        |
|2025-01-01  |6.0     |-73.5    |16.692193603515648 |14.357385253906273        |
|2025-01-01  |6.0     |-73.25   |15.932000732421898 |12.403039550781273        |
|2025-01-01  |6.0     |-73.0    |13.354058837890648 |8.955438232421898         |
|2025-01-01  |6.0     |-72.75   |11.741571044921898 |7.712701416015648         |
|2025-01-01  |6.0     |-72.5    |13.028192138671898 |9.906457519531273         |
|2025-01-01  |6.0     |-72.25   |16.921685791015648 |13.614221191406273        |
|2025-01-01  |6.0     |-72.0    |24.616906738281273 |17.602838134765648        |
|2025-01-01  |6.0     |-71.7

Faz a conversão de temperatura para percentual de umidade usando a equação de Magnus-Tetens.

https://en.wikipedia.org/wiki/Tetens_equation


In [12]:

# Constantes da equação de Magnus-Tetens.
A = 17.67
B = 243.5

df_umidade_Magnus_Tetens = (
    df_temp_ponto_orvalho
        .withColumn("indicador", F.lit("umidade"))
        .withColumn("unidade_medida", F.lit("percentual"))
        .withColumn("valor",
            F.round(  
                F.lit(100.0) * F.exp(
                    (
                        A * F.col("temp_ponto_orvalho_celsius") /
                        (F.col("temp_ponto_orvalho_celsius") + B)
                    )
                    -
                    (
                        A * F.col("temperatura_celsius") /
                        (F.col("temperatura_celsius") + B)
                    )
                )
            ,4)
        )
).drop("temperatura_celsius", "temp_ponto_orvalho_celsius")

df_umidade_Magnus_Tetens.show()

+------------+--------+---------+---------+--------------+-------+
|data_medicao|latitude|longitude|indicador|unidade_medida|  valor|
+------------+--------+---------+---------+--------------+-------+
|  2025-01-01|     6.0|    -74.0|  umidade|    percentual|89.2614|
|  2025-01-01|     6.0|   -73.75|  umidade|    percentual|88.0587|
|  2025-01-01|     6.0|    -73.5|  umidade|    percentual|86.0939|
|  2025-01-01|     6.0|   -73.25|  umidade|    percentual| 79.556|
|  2025-01-01|     6.0|    -73.0|  umidade|    percentual| 74.687|
|  2025-01-01|     6.0|   -72.75|  umidade|    percentual|76.3113|
|  2025-01-01|     6.0|    -72.5|  umidade|    percentual|81.3326|
|  2025-01-01|     6.0|   -72.25|  umidade|    percentual|80.8533|
|  2025-01-01|     6.0|    -72.0|  umidade|    percentual|64.9799|
|  2025-01-01|     6.0|   -71.75|  umidade|    percentual| 60.016|
|  2025-01-01|     6.0|    -71.5|  umidade|    percentual| 60.995|
|  2025-01-01|     6.0|   -71.25|  umidade|    percentual|59.0

In [14]:
df_umidade = \
    (df_umidade_Magnus_Tetens
        .select("data_medicao"
               ,"latitude"
               ,"longitude"
               ,"indicador"
               ,"valor"
               ,"unidade_medida"))

In [16]:
# df_umidade.toPandas().to_csv("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_umidade.csv", index=False)

df_umidade.toPandas().to_parquet("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_umidade.parquet")

c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [17]:
# df_csv = spark.read.csv("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_umidade_temperatura_precipitacao.csv", header=True, inferSchema=True)
# print("df_csv:", df_csv.count())
# df_csv.printSchema()
# df_csv.show(10,False)


df_parquet = spark.read.parquet("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_umidade.parquet")
print("df_csv:", df_parquet.count())
df_parquet.printSchema()
df_parquet.show(10,False)


df_csv: 379155
root
 |-- data_medicao: date (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- indicador: string (nullable = true)
 |-- valor: double (nullable = true)
 |-- unidade_medida: string (nullable = true)

+------------+--------+---------+---------+-------+--------------+
|data_medicao|latitude|longitude|indicador|valor  |unidade_medida|
+------------+--------+---------+---------+-------+--------------+
|2025-01-01  |6.0     |-74.0    |umidade  |89.2614|percentual    |
|2025-01-01  |6.0     |-73.75   |umidade  |88.0587|percentual    |
|2025-01-01  |6.0     |-73.5    |umidade  |86.0939|percentual    |
|2025-01-01  |6.0     |-73.25   |umidade  |79.556 |percentual    |
|2025-01-01  |6.0     |-73.0    |umidade  |74.687 |percentual    |
|2025-01-01  |6.0     |-72.75   |umidade  |76.3113|percentual    |
|2025-01-01  |6.0     |-72.5    |umidade  |81.3326|percentual    |
|2025-01-01  |6.0     |-72.25   |umidade  |80.8533|percentual 

In [18]:
# **** INCLUIR EXCLUSÃO DE ARQUIVOS (.zip e .nc)

for file in remover_arquivos:
    print("Arquivo:", file, end="")
    os.remove(file)
    print(" Removido com sucesso")


Arquivo: 48595421830250f6217c0f1557b75ae6.nc Removido com sucesso
